In [70]:
# Load just the headers of the full CSV to find the real column name
import pandas as pd

headers = pd.read_csv(
    '../Data-Folder/en.openfoodfacts.org.products.csv',
    sep='\t',
    nrows=0,           # load zero rows — headers only, very fast
    low_memory=False,
    on_bad_lines='skip'
)

print(f"Total columns in file: {len(headers.columns)}")
print()

# Find anything nutrition/score related
nutri_cols = [c for c in headers.columns if
              'nutri' in c.lower() or
              'score' in c.lower() or
              'grade' in c.lower()]

print("Nutrition/score related columns found:")
for c in nutri_cols:
    print(f"  {c}")

Total columns in file: 210

Nutrition/score related columns found:
  no_nutrition_data
  nutriscore_score
  nutriscore_grade
  environmental_score_score
  environmental_score_grade
  nutrient_levels_tags
  image_nutrition_url
  image_nutrition_small_url


In [71]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

   pandas  : 3.0.2
   numpy   : 2.4.4


## Load Dataset (500,000 rows)

In [72]:
FILENAME    = '../Data-Folder/en.openfoodfacts.org.products.csv'
SAMPLE_SIZE = 500_000

# Only load columns relevant to our analysis
COLS_NEEDED = [
    'product_name',
    'categories_tags',
    'ingredients_text',
    'energy_100g',
    'fat_100g',
    'saturated-fat_100g',
    'carbohydrates_100g',
    'sugars_100g',
    'fiber_100g',
    'proteins_100g',
    'salt_100g',
    'nutriscore_score'
]

print(f'Loading {SAMPLE_SIZE:,} rows from "{FILENAME}"...')
print('Please wait ~30–60 seconds...\n')

raw_df = pd.read_csv(
    FILENAME,
    sep='\t',
    usecols=lambda col: col in COLS_NEEDED,
    nrows=SAMPLE_SIZE,
    low_memory=False,
    on_bad_lines='skip'
)

print(f'   Rows    : {raw_df.shape[0]:,}')
print(f'   Columns : {raw_df.shape[1]}')
print(f'   Memory  : {raw_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
raw_df.head(3)

Loading 500,000 rows from "../Data-Folder/en.openfoodfacts.org.products.csv"...
Please wait ~30–60 seconds...

   Rows    : 500,000
   Columns : 12
   Memory  : 198.2 MB


,product_name,categories_tags,ingredients_text,nutriscore_score,energy_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g
0,Limonade artisanale a la rose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M&amp;M white,NaN,"Weizenmehl, Rapsöl, Speisesalz, 1,7% Meersalz,...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Chocolate n3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [73]:
# Print ALL column names in the dataset
print(f"Total columns: {len(raw_df.columns)}")
print()
for i, col in enumerate(sorted(raw_df.columns.tolist()), 1):
    print(f"  {i:>3}. {col}")

Total columns: 12

    1. carbohydrates_100g
    2. categories_tags
    3. energy_100g
    4. fat_100g
    5. fiber_100g
    6. ingredients_text
    7. nutriscore_score
    8. product_name
    9. proteins_100g
   10. salt_100g
   11. saturated-fat_100g
   12. sugars_100g


In [74]:
print(f'Shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns')
print()


desc = raw_df[['sugars_100g', 'proteins_100g', 'fat_100g',
               'carbohydrates_100g', 'fiber_100g', 'energy_100g']].describe().round(2)
print(desc)
print()


print('=== IMPOSSIBLE VALUES SPOTTED ===')
for col in ['sugars_100g', 'proteins_100g', 'fat_100g', 'carbohydrates_100g']:
    max_val = raw_df[col].max()
    min_val = raw_df[col].min()
    flag = ' <-- IMPOSSIBLE!' if max_val > 100 or min_val < 0 else ' OK'
    print(f'  {col:<25}  min: {min_val:>8.1f}  max: {max_val:>8.1f}{flag}')

Shape: 500,000 rows x 12 columns

         sugars_100g  proteins_100g    fat_100g  carbohydrates_100g  \
count      105210.00      108426.00   108283.00           108258.00   
mean        95061.12        9270.02      174.31              166.98   
std      30829855.50     3036945.57    53345.27            44008.11   
min             0.00           0.00        0.00                0.00   
25%             1.00           2.00        0.65                6.78   
50%             4.65           6.00        4.85               22.12   
75%            15.91          10.71       17.60               56.25   
max   10000000000.00  1000000000.00 17554003.53         14479774.25   

       fiber_100g  energy_100g  
count    77360.00    109257.00  
mean       132.76     10423.79  
std      35953.61   2926529.54  
min         -0.15      -547.00  
25%          0.00       416.84  
50%          1.67      1099.11  
75%          3.60      1686.67  
max   10000000.00 966426848.38  

=== IMPOSSIBLE VALUES SPOTTE

## Handle missing values

In [75]:
missing     = raw_df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(raw_df) * 100).round(1)

audit = pd.DataFrame({
    'Missing Count': missing,
    'Missing %'    : missing_pct
})
print(audit.to_string())

                    Missing Count  Missing %
fiber_100g                 422640      84.50
salt_100g                  401901      80.40
saturated-fat_100g         396734      79.30
sugars_100g                394790      79.00
carbohydrates_100g         391742      78.30
fat_100g                   391717      78.30
proteins_100g              391574      78.30
energy_100g                390743      78.10
nutriscore_score           265484      53.10
ingredients_text           232229      46.40
categories_tags            231411      46.30
product_name                15692       3.10


## Dropping rows with null values

In [76]:
#Drop rows missing critical columns
CRITICAL_COLS = ['product_name', 'sugars_100g', 'proteins_100g', 'categories_tags']

rows_before = len(raw_df)
df = raw_df.dropna(subset=CRITICAL_COLS).copy()
rows_after  = len(df)

print(f'Rows before null removal : {rows_before:,}')
print(f'Rows after null removal  : {rows_after:,}')
print(f'Rows removed             : {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.1f}%)')
print(f'\nNULL values handled')

Rows before null removal : 500,000
Rows after null removal  : 50,033
Rows removed             : 449,967 (90.0%)

NULL values handled


## Filter Biologically Impossible Values

All nutritional values are measured **per 100g of product**.
Therefore no single nutrient can exceed 100g per 100g , that is physically impossible.
Values outside the range `[0, 100]` are data entry errors and must be removed.

In [77]:
for col in ['sugars_100g', 'proteins_100g', 'fat_100g', 'carbohydrates_100g']:
    bad = df[(df[col] < 0) | (df[col] > 100)][col].count()
    print(f'  {col:<25} : {bad:,} impossible values')

  sugars_100g               : 58 impossible values
  proteins_100g             : 49 impossible values
  fat_100g                  : 96 impossible values
  carbohydrates_100g        : 178 impossible values


In [78]:
rows_before = len(df)

df = df[
    (df['sugars_100g'].between(0, 100))        &
    (df['proteins_100g'].between(0, 100))      &
    (df['fat_100g'].fillna(0).between(0, 100)) &
    (df['carbohydrates_100g'].fillna(0).between(0, 100))
]

rows_after = len(df)

print(f'Rows before outlier removal : {rows_before:,}')
print(f'Rows after outlier removal  : {rows_after:,}')
print(f'Impossible values removed   : {rows_before - rows_after:,}')
print(f'Biologically impossible values removed')

Rows before outlier removal : 50,033
Rows after outlier removal  : 49,780
Impossible values removed   : 253
Biologically impossible values removed


## Remove Duplicate Rows

A duplicate is defined as a row with the same `product_name` AND
the same `sugars_100g` AND `proteins_100g`.
We keep the first occurrence and drop the rest.

In [79]:
rows_before = len(df)

df = df.drop_duplicates(
    subset=['product_name', 'sugars_100g', 'proteins_100g'],
    keep='first'
)

df = df.reset_index(drop=True)
rows_after = len(df)

print(f'Rows before deduplication : {rows_before:,}')
print(f'Rows after deduplication  : {rows_after:,}')
print(f'Duplicates removed        : {rows_before - rows_after:,}')
print(f' Duplicates removed')

Rows before deduplication : 49,780
Rows after deduplication  : 47,495
Duplicates removed        : 2,285
 Duplicates removed


In [81]:
EXPORT_COLS = [
    'product_name',
    'categories_tags',
    'ingredients_text',
    'energy_100g',
    'fat_100g',
    'saturated-fat_100g',
    'carbohydrates_100g',
    'sugars_100g',
    'fiber_100g',
    'proteins_100g',
    'salt_100g',
    'nutriscore_score'
]

# Keep only columns that actually exist in df (some may be missing from dataset)
available_cols = [c for c in EXPORT_COLS if c in df.columns]
missing_cols   = [c for c in EXPORT_COLS if c not in df.columns]


OUTPUT_FILE = '../Master-Data/clean_food_data.csv'

df.to_csv(OUTPUT_FILE, index=False)

print(f'   File    : {OUTPUT_FILE}')
print(f'   Rows    : {len(df):,}')
print(f'   Columns : {len(df.columns)}')
print(f'   Columns  : {len(available_cols)}')
print(f'   Exported : {available_cols}')
if missing_cols:
    print(f'Missing from dataset: {missing_cols}')
else:
    print(f'All columns including nutriscore_score exported')

   File    : ../Master-Data/clean_food_data.csv
   Rows    : 47,495
   Columns : 12
   Columns  : 12
   Exported : ['product_name', 'categories_tags', 'ingredients_text', 'energy_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'nutriscore_score']
All columns including nutriscore_score exported
